# Supervised activity-recognition comparison

Compare logistic regression, a 1D CNN, a GRU, and a CNN–GRU on one fixed subject-separated validation group. The neural models use three seeds; the deterministic logistic baseline is fitted once.

**This notebook reports development validation results. It does not score the official test set or demonstrate phone performance.** The full protocol is in [the English guide](../../docs/supervised-module.md). Inputs follow [preprocessing v1](../../contracts/preprocessing-v1.md).

## Environment and run mode

Select the project environment as the kernel. From the repository root, install PyTorch and the optional dependencies:

```text
python -m pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cpu
python -m pip install -e "./ml[supervised,notebook]"
python -m ipykernel install --sys-prefix --name harss --display-name "HARSS (.venv)"
```

Prepare Module 1's data before training. Set `RUN_TRAINING=False` to view the committed results without opening any signal arrays. Keep one writer per output directory; do not start a second training process while the command-line experiment is running.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import torch
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ml/configs/subjects-v1.json").is_file())
DATA_DIR = ROOT / "ml/data/prepared/uci-har-v1"
CONFIG_PATH = ROOT / "ml/configs/supervised-v1.json"
OUTPUT_DIR = ROOT / "ml/outputs/supervised-v1"
PUBLISHED_DIR = ROOT / "docs/experiments/supervised-v1"
RUN_TRAINING = True
WRITE_REPORT = False

from harss_supervised.data import load_training_data
from harss_supervised.experiment import load_config, run_comparison
from harss_supervised.features import FEATURE_NAMES, statistical_features
from harss_supervised.models import build_model
from harss_supervised.reporting import (
    comparison_figure, confusion_figure, learning_figure, package_exports, publish_report, summary_table,
)

config = load_config(CONFIG_PATH)
display(pd.DataFrame([{
    "Python": platform.python_version(), "NumPy": np.__version__,
    "scikit-learn": sklearn.__version__, "PyTorch": str(torch.__version__),
    "experiment_device": config["device"], "CPU_threads": config["cpu_threads"],
}]))

## Fixed data and leakage checks

There are 5,658 training windows from 16 subjects and 1,694 validation windows from five different subjects. Each has 128 time points and six channels: total acceleration including gravity, then angular velocity. The input statistics were fitted only on training subjects.

The comparison loader verifies training/validation hashes and subject separation. It never opens `test.npz`. In recorded-results mode, this cell opens no signal arrays.

In [ ]:
if RUN_TRAINING:
    data = load_training_data(DATA_DIR)
    display(pd.DataFrame([
        {"partition": name, "windows": len(part.x),
         "subjects": sorted(map(int, set(part.subject_id))),
         "window_shape": str(part.x.shape[1:]), "dtype": str(part.x.dtype)}
        for name, part in [("train", data.train), ("validation", data.validation)]
    ]))
    assert not data.provenance["test_arrays_opened"]
    assert not data.provenance["test_scores_computed"]
    print("Training/validation lineage verified; test arrays were not opened.")
else:
    data = None
    print("Recorded-results mode: no training, validation, or test arrays opened.")

## Representations and architectures

The logistic baseline computes five statistics for each channel: mean, population standard deviation, minimum, maximum, and root mean square. Its 30 features receive a separate scaler fitted on training features only. It does not use the dataset's 561-feature vectors.

The CNN has two convolution/pooling blocks followed by global mean/max pooling. The GRU reads all 128 points. The CNN–GRU uses the same convolutional front end, then a GRU over 32 pooled time points. All neural classifiers return six logits in the same class order.

In [ ]:
display(pd.DataFrame([
    {"model": name, "trainable_parameters": sum(p.numel() for p in build_model(name, config).parameters())}
    for name in ("cnn", "gru", "cnn_gru")
]))
print("Statistical feature count:", len(FEATURE_NAMES))
print("Feature-order example:", FEATURE_NAMES[:5])
if data is not None:
    assert statistical_features(data.train.x[:2]).shape == (2, 30)
    print("Feature extraction verified on two training windows.")
display(pd.Series({key: config[key] for key in [
    "seeds", "epochs", "patience", "batch_size", "learning_rate", "weight_decay",
    "dropout", "selection_metric", "tie_break", "test_scoring",
]}, name="Frozen protocol").to_frame())

## Train or reuse verified completed fits

Each neural architecture runs seeds 570, 571, and 572 for at most 30 epochs, with the same AdamW settings and early-stopping rule. The best validation macro-F1 checkpoint is retained; exact epoch ties keep the earlier epoch. The logistic baseline is one deterministic fit.

Before formal training, a separate check memorizes 24 training windows with dropout disabled. These mechanics checks do not estimate generalization. The runner saves a code/configuration/data/software signature. Matching completed fits are checksum-verified and reused; changed signatures require a new output directory.

In [ ]:
if RUN_TRAINING:
    report = run_comparison(DATA_DIR, CONFIG_PATH, OUTPUT_DIR)
else:
    report = json.loads((PUBLISHED_DIR / "results.json").read_text(encoding="utf-8"))

assert report["status"] == "complete"
assert report["evaluation_partition"] == "validation"
assert not report["test_scores_computed"]
assert len(report["runs"]) == 10
print("Complete:", len(report["runs"]), "formal fits; test scoring:", report["test_scores_computed"])
print("Training source hash:", report["provenance"]["training_code_sha256"])
display(pd.DataFrame([report["provenance"]["software"]]))

## Validation comparison

Mean ± sample standard deviation is shown across the three neural seeds. Logistic regression has one deterministic score and no across-seed dispersion estimate. These are scores on the same five validation subjects, not independent subject-split repetitions or final test results.

In [ ]:
def mean_sd(mean, std):
    return f"{mean:.4f}" if std is None else f"{mean:.4f} ± {std:.4f}"

comparison = []
for row in summary_table(report):
    comparison.append({
        "Model": row["model"], "Runs": row["n_runs"],
        "Validation accuracy": mean_sd(row["validation_accuracy_mean"], row["validation_accuracy_sample_std"]),
        "Validation macro-F1": mean_sd(row["validation_macro_f1_mean"], row["validation_macro_f1_sample_std"]),
        "Parameters": row["parameters"],
        "Mean training (s)": round(row["mean_training_seconds"], 2),
        "Desktop median (ms)": round(row["mean_desktop_median_ms"], 3),
    })
display(pd.DataFrame(comparison).set_index("Model"))
comparison_figure(report)
plt.show()

## Class-level behavior

Mean class recall averages the available runs of each architecture. Confusion matrices below use the predeclared first seed (570), rather than selecting the most favorable seed for the figure. Rows are true classes; columns are predictions; the color scale is shared and rows sum to one.

In [ ]:
recalls = {}
for model_name in config["models"]:
    group = [run for run in report["runs"] if run["model"] == model_name]
    recalls[model_name] = {
        name: np.mean([run["validation"]["per_class"][name]["recall"] for run in group])
        for name in report["classes"]
    }
display(pd.DataFrame(recalls).rename_axis("Activity").round(4))
confusion_figure(report)
plt.show()

## Learning curves and stopping decisions

Each curve is one seed. A circle marks the checkpoint selected by validation macro-F1. Shorter curves reflect the same six-epoch patience rule. The formal fit uses the normal dropout setting, unlike the separate mechanics check.

In [ ]:
learning_figure(report)
plt.show()
display(pd.DataFrame([
    {"Model": r["model"], "Seed": r["seed"], "Epochs run": r["epochs_run"],
     "Selected epoch": r["best_epoch"], "Validation macro-F1": r["validation"]["macro_f1"]}
    for r in report["runs"] if r["stochastic_training"]
]).round(4))

## Training checks and desktop export parity

All saved checkpoints are reconstructed and checked against their validation predictions. For each neural architecture, the best validation seed is also exported to `.pt2`, saved, reloaded, and compared on six training windows at batch size 1. This validates the desktop representation and output order. It does not establish Android runtime compatibility.

The table of memorization checks below concerns training mechanics only; those accuracy values must not be reported as generalization performance.

In [ ]:
display(pd.DataFrame(report["sanity_checks"]).T[
    ["windows", "steps", "initial_loss", "final_loss", "training_accuracy"]
])
display(pd.DataFrame([
    {"Model": c["model"], "Export seed": c["seed"], "Format": c["format"],
     "Windows checked": c["training_windows_checked"],
     "Maximum logit error": c["maximum_absolute_logit_error"], "Passed": c["passed"]}
    for c in report["export_checks"]
]))
assert all(r["checkpoint_reload_matches"] for r in report["runs"])
assert all(c["passed"] for c in report["export_checks"])
print("All checkpoint and desktop export checks passed.")

## Development choice and limits

Architectures are ranked by mean validation macro-F1, then fewer classifier parameters, then model name. Validation selected both epochs and architecture, so these scores are development estimates. Desktop timing uses 10 warmup and 50 timed CPU calls from an already-standardized window; it excludes raw-sensor processing and is not a phone measurement.

In [ ]:
display(Markdown(
    "Highest mean validation macro-F1: **" +
    report["selection"]["best_model_by_mean_validation_macro_f1"] + "**.\n\n" +
    "Neural candidate for the next experiment: **" +
    report["selection"]["best_neural_model_by_mean_validation_macro_f1"] + "**."
))
for limitation in report["limitations"]:
    print("-", limitation)

## Save a reviewable report

The committed reference report is under `docs/experiments/supervised-v1/`. To publish the current run's aggregate JSON, CSV, English summary, and PNG/SVG figures there, set `WRITE_REPORT=True`. Weights, full predictions, and source windows remain in ignored local directories. Running this cell with the default switch does not replace the reference report.

In [ ]:
if WRITE_REPORT:
    from harss_data.prepare import write_json
    publish_report(report, PUBLISHED_DIR)
    if RUN_TRAINING:
        bundles = package_exports(report, DATA_DIR, OUTPUT_DIR)
        write_json(PUBLISHED_DIR / "export-manifests.json", {"exports": bundles})
    print("Saved aggregate report to docs/experiments/supervised-v1/")
else:
    print("Reference report unchanged. Set WRITE_REPORT=True to save the current report.")

## Reproduction and references

Run from the repository root with the selected environment:

```text
python -m unittest discover -s ml/tests -v
python -m harss_supervised
python ml/scripts/execute_notebook.py
```

The first command validates data boundaries, metrics, gradients, checkpoint reloads, and exports. The experiment reuses checksum-verified completed fits. To conduct a separate comparison, choose a new output path and record the changed configuration before inspecting results.

- [Experiment protocol and API references](../../docs/supervised-module.md)
- [Dataset attribution and preparation](../../docs/data-module.md)
- [Machine-readable reference results](../../docs/experiments/supervised-v1/results.json)

The official test set remains unscored for the later frozen comparison. Self-supervised learning, distillation, phone recording, and Android evaluation are separate modules.